In [6]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "vscode"

from pathlib import Path

# =============================================================================
# Project directories
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = PROJECT_ROOT / "figures"

SAVE_HTML = True
SHOW_FIGURE = False

In [7]:
clean_dataset_file = RAW_DATA_DIR / "caes_dataset_clean.csv"
df_clean = pd.read_csv(clean_dataset_file, index_col='timestamp')
df_clean.index = pd.to_datetime(df_clean.index)
#
noisy_dataset_file = RAW_DATA_DIR / "caes_dataset_noisy.csv"
df_noisy = pd.read_csv(noisy_dataset_file, index_col='timestamp')
df_noisy.index = pd.to_datetime(df_noisy.index)

In [8]:
# =============================================================================
# Dataset overview
# =============================================================================
df = df_noisy.copy()
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 17280 entries, 2026-01-01 00:00:00 to 2026-01-02 23:59:50
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   operating_mode       17280 non-null  str    
 1   cycle_id             17280 non-null  int64  
 2   ambient_temperature  17203 non-null  float64
 3   tank_pressure        17280 non-null  float64
 4   tank_temperature     17231 non-null  float64
 5   compressor_power     17280 non-null  float64
 6   turbine_power        17280 non-null  float64
 7   mass_flow            17280 non-null  float64
dtypes: float64(6), int64(1), str(1)
memory usage: 1.3 MB


# Operational KPI Calculation

After completing the data quality assessment, the next step is to evaluate
the operational performance of the energy storage asset.

Operational Key Performance Indicators (KPIs) provide a quantitative
assessment of the plant behaviour by converting raw sensor measurements
into meaningful engineering metrics.

The main objectives of this section are:

- quantify the electrical energy consumed during charging;
- quantify the electrical energy produced during discharging;
- evaluate the round-trip efficiency of the storage cycle;
- analyse plant behaviour according to the operating mode.

The calculated KPIs are based on the generated synthetic measurements and
are intended to demonstrate an energy analytics workflow rather than
reproduce the exact performance of a commercial CAES plant.

---

## Energy calculation

The dataset contains instantaneous power measurements:

- compressor power during charging [kW];
- turbine power during discharging [kW].

To calculate energy, power values are integrated over the sampling period.

The energy contribution of each measurement interval is calculated as:

$
E_i = P_i \cdot \Delta t
$

where:

- \(E_i\) is the energy associated with the timestep [kWh];
- \(P_i\) is the measured power [kW];
- \(\Delta t\) is the sampling interval expressed in hours.

For this dataset:

$
\Delta t = \frac{10}{3600} \text{ h}
$

since measurements are sampled every 10 seconds.

---

## Energy input

During charging, electrical energy is consumed by the compressor to store
energy in the compressed air reservoir.

The total energy input is calculated as:

$
E_{in}=\int P_{compressor}(t)dt
$

where:

- \(E_{in}\) is the electrical energy consumed during charging [kWh];
- \(P_{compressor}\) is compressor electrical power [kW].

---

## Energy output

During discharging, stored compressed air is converted back into electrical
energy through the turbine.

The total energy output is calculated as:


$E_{out}=\int P_{turbine}(t)dt$


where:

- \(E_{out}\) is the generated electrical energy [kWh];
- \(P_{turbine}\) is turbine electrical power [kW].

---

## Round-trip efficiency

The round-trip efficiency (RTE) is the main performance indicator for an
energy storage system.

It represents the fraction of electrical energy recovered compared with
the energy initially supplied:


$RTE=\frac{E_{out}}{E_{in}}\times100$


A higher value indicates a more efficient storage cycle.

The calculated value depends on the simplified assumptions adopted in the
synthetic model, particularly the compressor and turbine power models.

Therefore, this KPI is used to demonstrate the calculation methodology and
monitoring workflow rather than to reproduce the exact efficiency of a
commercial CAES facility.

---

## Performance by operating mode

In addition to global energy indicators, KPIs are calculated separately
for each operating condition:

- STANDBY
- CHARGING
- DISCHARGING

This allows the analysis of:

- operating time distribution;
- energy consumption during charging;
- energy production during discharge.

Such mode-based analysis is commonly used in industrial asset monitoring
to identify operational patterns and evaluate system performance.

In [ ]:
# =============================================================================
# KPI calculation parameters
# =============================================================================

SAMPLING_TIME_SECONDS = 10

TIME_STEP_HOURS = (
    SAMPLING_TIME_SECONDS / 3600
)

# =============================================================================
# Energy per timestep
# =============================================================================

df["compressor_energy_kwh"] = (
    df["compressor_power"] *
    TIME_STEP_HOURS
)

df["turbine_energy_kwh"] = (
    df["turbine_power"] *
    TIME_STEP_HOURS
)
# =============================================================================
# Total energy KPIs
# =============================================================================

energy_input = (
    df["compressor_energy_kwh"]
    .sum()
)

energy_output = (
    df["turbine_energy_kwh"]
    .sum()
)

print(f"Energy consumed: {energy_input:.2f} kWh")
print(f"Energy produced: {energy_output:.2f} kWh")

# =============================================================================
# Round Trip Efficiency
# =============================================================================

round_trip_efficiency = (
    energy_output /
    energy_input *
    100
)

print(f"Round Trip Efficiency: {round_trip_efficiency:.1f}%")

# =============================================================================
# Global KPI summary
# =============================================================================

operating_hours = len(df) * TIME_STEP_HOURS

number_of_cycles = df["cycle_id"].nunique()

summary_kpi = pd.DataFrame({
    "KPI": [
        "Operating hours",
        "Number of cycles",
        "Energy consumed",
        "Energy produced",
        "Round-trip efficiency"
    ],
    "Value": [
        round(operating_hours, 2),
        number_of_cycles,
        round(energy_input, 2),
        round(energy_output, 2),
        round(round_trip_efficiency, 1)
    ],
    "Unit": [
        "h",
        "-",
        "kWh",
        "kWh",
        "%"
    ]
})

display(summary_kpi)

# =============================================================================
# Operating mode KPI
# =============================================================================

mode_kpi = (
    df
    .groupby("operating_mode")
    .agg(
        operating_hours=(
            "operating_mode",
            lambda x: len(x) * TIME_STEP_HOURS
        ),

        compressor_energy_kwh=(
            "compressor_energy_kwh",
            "sum"
        ),

        turbine_energy_kwh=(
            "turbine_energy_kwh",
            "sum"
        ),

        average_pressure=(
            "tank_pressure",
            "mean"
        ),

        average_temperature=(
            "tank_temperature",
            "mean"
        )
    )
)

mode_kpi["operating_percentage"] = (
    mode_kpi["operating_hours"]
    / operating_hours
    * 100
)

display(mode_kpi.round(2))

Energy consumed: 1430.11 kWh
Energy produced: 452.76 kWh
Round Trip Efficiency: 31.7%


,KPI,Value,Unit
0,Operating hours,48.00,h
1,Number of cycles,4.00,-
2,Energy consumed,1430.11,kWh
3,Energy produced,452.76,kWh
4,Round-trip efficiency,31.70,%


,operating_hours,compressor_energy_kwh,turbine_energy_kwh,average_pressure,average_temperature,operating_percentage
operating_mode,,,,,,
CHARGING,12.0,1429.89,0.00,245.48,57.25,25.00
DISCHARGING,8.0,0.84,452.76,230.46,10.71,16.67
STANDBY,28.0,-0.62,0.00,224.16,20.66,58.33


In [14]:
# =============================================================================
# Cycle KPI
# =============================================================================

cycle_kpi = (
    df
    .groupby("cycle_id")
    .agg(
        energy_input_kwh=(
            "compressor_energy_kwh",
            "sum"
        ),

        energy_output_kwh=(
            "turbine_energy_kwh",
            "sum"
        ),

        duration_hours=(
            "cycle_id",
            lambda x: len(x) * TIME_STEP_HOURS
        )
    )
)

cycle_kpi["round_trip_efficiency"] = (
    cycle_kpi["energy_output_kwh"]
    /
    cycle_kpi["energy_input_kwh"]
    * 100
)

display(cycle_kpi.round(2))

,energy_input_kwh,energy_output_kwh,duration_hours,round_trip_efficiency
cycle_id,,,,
1,0.00,0.00,6.0,NaN
2,736.92,230.30,16.0,31.25
3,469.09,222.46,17.0,47.42
4,224.11,0.00,9.0,0.00


In [15]:
fig = px.bar(
    mode_kpi.reset_index(),
    x="operating_mode",
    y=[
        "compressor_energy_kwh",
        "turbine_energy_kwh"
    ],
    barmode="group",
    labels={
        "value": "Energy [kWh]",
        "operating_mode": "Operating Mode"
    },
    title="Energy Contribution by Operating Mode"
)

fig.update_layout(
    template="plotly_white",
    legend_title=""
)

fig.show()

In [16]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=4,
    specs=[[{"type":"indicator"}]*4]
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=energy_input,
        title={"text":"Energy In"}
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=energy_output,
        title={"text":"Energy Out"}
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=round_trip_efficiency,
        number={"suffix":" %"},
        title={"text":"RTE"}
    ),
    row=1,
    col=3
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=number_of_cycles,
        title={"text":"Cycles"}
    ),
    row=1,
    col=4
)

fig.update_layout(
    height=220,
    template="plotly_white",
    title="Operational KPI Summary"
)

fig.show()

In [17]:
mode_map = {
    "STANDBY":0,
    "CHARGING":1,
    "DISCHARGING":2
}

df["mode_numeric"] = (
    df["operating_mode"]
    .map(mode_map)
)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.10, 0.25, 0.25, 0.25, 0.15],
    subplot_titles=(
        "",
        "Tank Pressure",
        "Tank Temperature",
        "Electrical Power",
        "Operating Mode"
    )
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["tank_pressure"],
        name="Pressure",
        line=dict(color="royalblue")
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["tank_temperature"],
        name="Temperature",
        line=dict(color="firebrick")
    ),
    row=3,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["compressor_power"],
        name="Compressor",
        line=dict(color="darkorange")
    ),
    row=4,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["turbine_power"],
        name="Turbine",
        line=dict(color="purple")
    ),
    row=4,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["mode_numeric"],
        name="Operating Mode",
        line=dict(color="black", width=2)
    ),
    row=5,
    col=1
)

fig.update_yaxes(
    tickvals=[0, 1, 2],
    ticktext=[
        "Standby",
        "Charging",
        "Discharging"
    ],
    row=5,
    col=1
)

dashboard_title = (
    "<b>Operational Performance Dashboard</b><br>"
    f"Energy In: <b>{energy_input:.1f} kWh</b> | "
    f"Energy Out: <b>{energy_output:.1f} kWh</b> | "
    f"RTE: <b>{round_trip_efficiency:.1f}%</b> | "
    f"Cycles: <b>{number_of_cycles}</b>"
)

fig.update_layout(

    title=dashboard_title,
    template="plotly_white",
    height=950,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(
        l=50,
        r=30,
        t=100,
        b=40
    )
)

fig.update_xaxes(
    showspikes=True,
    spikemode="across",
    spikesnap="cursor",
    spikethickness=1,
    spikedash="dot"
)

fig.update_xaxes(
    title="Time",
    row=5,
    col=1
)

if SAVE_HTML:
    filename = "dashboard.html"
    fig.write_html(FIGURES_DIR / filename,
                   include_plotlyjs="cdn")

if SHOW_FIGURE:
    fig.show()